# CROHME Colab Quickstart

This notebook supports two Colab workflows:

- `inference_only`: run direct evaluation for the base model on `2014`, `2016`, and `2019`
- `lora_train_eval`: fine-tune with LoRA on the `train` split, then evaluate on `2014`, `2016`, and `2019`

Use a GPU runtime in Colab before running the cells.

This notebook runs the local pipeline only: exact match, CER, edit score, and BLEU-4. UniMERNet CDM is intentionally decoupled from the default Colab flow.

In [ ]:
REPO_URL = "https://github.com/Zeric-li/HME.git"
WORKDIR = "/content/HME"

!rm -rf "$WORKDIR"
!git clone --depth 1 --branch main "$REPO_URL" "$WORKDIR"
%cd "$WORKDIR"

In [ ]:
!python -m pip install -U pip
!pip install -r requirements.txt
!pip install -U "git+https://github.com/huggingface/transformers"

In [ ]:
TASK = "inference_only"  # or "lora_train_eval"
MAX_EVAL_SAMPLES = 32     # set to None for full evaluation
MAX_TRAIN_SAMPLES = 128   # used only for lora_train_eval; set to None for full training
OUTPUT_ROOT = "/content/outputs"

assert TASK in {"inference_only", "lora_train_eval"}

In [ ]:
from pathlib import Path
import importlib.util
import torch
import yaml

def resolve_colab_runtime_overrides() -> dict:
    overrides = {
        'torch_dtype': 'float16',
        'attn_implementation': 'sdpa',
    }
    if not torch.cuda.is_available():
        return overrides

    major, _ = torch.cuda.get_device_capability()
    has_flash_attn = importlib.util.find_spec('flash_attn') is not None
    if major >= 8:
        overrides['torch_dtype'] = 'bfloat16'
    if has_flash_attn:
        overrides['attn_implementation'] = 'flash_attention_2'
    return overrides

base_config_path = Path("configs/crohme_inference_pipeline.yaml" if TASK == "inference_only" else "configs/crohme_lora_pipeline.yaml")
with open(base_config_path, "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

config.update(resolve_colab_runtime_overrides())
config["output_dir"] = str(Path(OUTPUT_ROOT) / Path(config["output_dir"]).name)
config["max_eval_samples"] = MAX_EVAL_SAMPLES
if TASK == "lora_train_eval":
    config["max_train_samples"] = MAX_TRAIN_SAMPLES
    config["per_device_train_batch_size"] = 1
    config["per_device_eval_batch_size"] = 1
    config["eval_batch_size"] = 1
    config["gradient_accumulation_steps"] = max(int(config.get("gradient_accumulation_steps", 8)), 8)

print('Resolved Colab runtime overrides:')
print({
    'torch_dtype': config['torch_dtype'],
    'attn_implementation': config['attn_implementation'],
})

colab_config_path = Path("configs") / f"colab_{TASK}.yaml"
with open(colab_config_path, "w", encoding="utf-8") as handle:
    yaml.safe_dump(config, handle, sort_keys=False)

print(f"Wrote Colab config to: {colab_config_path}")
print(yaml.safe_dump(config, sort_keys=False))

In [ ]:
!nvidia-smi

In [ ]:
!python --version
!python -m scripts.evaluate_predictions --help >/dev/null
!python -m scripts.export_unimernet_cdm_input --help >/dev/null
!python -m scripts.generate_eval_report_figures --help >/dev/null

In [ ]:
from pathlib import Path
import subprocess

config_path = str(Path("configs") / f"colab_{TASK}.yaml")
script_path = "scripts/run_inference_pipeline.sh" if TASK == "inference_only" else "scripts/run_lora_pipeline.sh"
subprocess.run(["bash", script_path, config_path], check=True)

In [ ]:
from pathlib import Path
import json
import subprocess
import yaml
from IPython.display import Image, display


def visualize_results(task: str) -> Path:
    config_path = Path('configs') / f'colab_{task}.yaml'
    cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    root = Path(cfg['output_dir'])
    results_dir = root / 'overall_results' if task == 'inference_only' else root / 'checkpoint-final' / 'overall_results'

    subprocess.run(
        ['python', '-m', 'scripts.generate_eval_report_figures', '--results-dir', str(results_dir)],
        check=True,
    )

    for metrics_path in sorted(root.glob('**/metrics.json')):
        print(f"\n=== {metrics_path} ===")
        with open(metrics_path, 'r', encoding='utf-8') as handle:
            print(json.dumps(json.load(handle), indent=2))

    fig_dir = results_dir / 'report_figures'
    print(f'\nReport figures written to: {fig_dir}')
    for fig_path in sorted(fig_dir.glob('*.png')):
        print(f"\n=== {fig_path.name} ===")
        display(Image(filename=str(fig_path)))

    return fig_dir

visualize_results(TASK)